# Assignment 8 – Build the Intent Dataset for My Hotel Booking Chatbot

**Student:** Rangira Rurangwa David  
**Course:** Chatbots Development (AITCD001), Week 8  
**Project:** Provisional Hotel Booking Chatbot  
**Dataset status:** Synthetic prototype; original Portfolio Project 1 rules have not been verified.

## Data and project limitations

I created all 180 messages in this prototype as synthetic examples. I did not collect them from real people, and none came from a UNMATCHED log or Step 10 local-phrasing records. I did not have the original Portfolio Project 1 Hotel Booking Chatbot source code available for verification when preparing this notebook. For that reason, the six intents and keyword rules in this notebook are provisional examples. The results should not be described as verified functionality or baseline performance of my original chatbot.

## Step 1. The chatbot and its labels

For this prototype, I designed a Hotel Booking Chatbot intent dataset with six labels:

- `room_availability`: asks whether a room or room type is available.
- `room_booking`: asks to make a new reservation.
- `booking_modification`: asks to change an existing reservation.
- `booking_cancellation`: asks to cancel an existing reservation.
- `hotel_amenities`: asks about hotel facilities or services.
- `out_of_scope`: asks for something outside the five hotel topics above.

These labels form a proposed intent set for this assignment. I have not verified them against the code from Portfolio Project 1, so I treat them as provisional.

## Step 2. Labelling guideline

I applied the following rules consistently when assigning labels. I considered the meaning and main action in the complete message rather than relying on a single keyword.

1. I used `booking_cancellation` when the main request was to cancel, remove, withdraw, or stop an existing reservation.
2. I used `booking_modification` when the request was to change, update, extend, move, or amend an existing booking without cancelling it.
3. I used `room_availability` when the message asked whether a room was vacant, free, open, or available. An availability question alone was not treated as a booking.
4. I used `hotel_amenities` for questions about hotel facilities or services such as Wi-Fi, breakfast, parking, a pool, a gym, or laundry.
5. I used `room_booking` when the message requested a new room reservation.
6. I used `out_of_scope` when the main request was unrelated to the five hotel topics.
7. When a message mentioned more than one topic, I labelled its main requested action. Cancellation took priority over modification, and asking about availability did not count as making a booking.
8. I preserved the message text as written, including spelling, abbreviations, Kinyarwanda, mixed-language wording, and SMS-style phrasing. I did not translate or silently correct messages.

## Step 3. The dataset

I stored the labelled messages in `intents.csv`. The `source` column marks every row as `synthetic`. The `language_style` column records the intended language or style coverage for the examples; it does not mean that the messages were collected from speakers.

The dataset contains 180 messages: 30 for each of the six labels.

In [1]:
import pandas as pd
import re
from collections import Counter
from pathlib import Path
from sklearn.model_selection import train_test_split

DATA_PATH = Path("intents.csv")
df = pd.read_csv(DATA_PATH)
APPROVED_LABELS = {
    "room_availability", "room_booking", "booking_modification",
    "booking_cancellation", "hotel_amenities", "out_of_scope"
}
print("Rows:", len(df))
print("Labels:", sorted(df["intent"].unique()))
print(df.groupby("intent").size().to_string())

Rows: 180
Labels: ['booking_cancellation', 'booking_modification', 'hotel_amenities', 'out_of_scope', 'room_availability', 'room_booking']
intent
booking_cancellation    30
booking_modification    30
hotel_amenities         30
out_of_scope            30
room_availability       30
room_booking            30


## Step 4. Quality report

I used the checks below to assess label coverage, minimum class size, class balance, duplicate text, approved labels, and language-style coverage. All rows are synthetic, so the requirement for at least 20 genuine messages from real people is not satisfied by this dataset.

In [2]:
counts = df["intent"].value_counts()
duplicates = int(df["text"].duplicated().sum())
unknown_labels = sorted(set(df["intent"]) - APPROVED_LABELS)
minimum = int(counts.min())
maximum = int(counts.max())
balance_ratio = maximum / minimum if minimum else float("inf")
style_coverage = df.groupby("intent")["language_style"].apply(set)
required_styles = {"Kinyarwanda", "mixed", "SMS-style"}
coverage_ok = all(required_styles.issubset(s) for s in style_coverage)
quality_checks = {
    "At least 30 messages per label": minimum >= 30,
    "Largest/smallest label ratio <= 1.5": balance_ratio <= 1.5,
    "No duplicate messages": duplicates == 0,
    "Only approved labels": len(unknown_labels) == 0,
    "Every label has Kinyarwanda, mixed and SMS-style examples": coverage_ok,
    "All rows transparently marked synthetic": set(df["source"]) == {"synthetic"},
}
for name, passed in quality_checks.items():
    print(f"{'PASS' if passed else 'FAIL'} | {name}")
print("Unique messages:", df["text"].nunique())
print("Unknown labels:", unknown_labels)
print("Label balance ratio:", round(balance_ratio, 2))
print("Genuine people messages:", int((df["source"] == "real_person").sum()))
print("Synthetic messages:", int((df["source"] == "synthetic").sum()))
print("Language-style coverage by label:")
for label, styles in style_coverage.items():
    print(label, sorted(styles))

PASS | At least 30 messages per label
PASS | Largest/smallest label ratio <= 1.5
PASS | No duplicate messages
PASS | Only approved labels
PASS | Every label has Kinyarwanda, mixed and SMS-style examples
PASS | All rows transparently marked synthetic
Unique messages: 180
Unknown labels: []
Label balance ratio: 1.0
Genuine people messages: 0
Synthetic messages: 180
Language-style coverage by label:
booking_cancellation ['English', 'Kinyarwanda', 'SMS-style', 'mixed']
booking_modification ['English', 'Kinyarwanda', 'SMS-style', 'mixed']
hotel_amenities ['English', 'Kinyarwanda', 'SMS-style', 'mixed']
out_of_scope ['English', 'Kinyarwanda', 'SMS-style', 'mixed']
room_availability ['English', 'Kinyarwanda', 'SMS-style', 'mixed']
room_booking ['English', 'Kinyarwanda', 'SMS-style', 'mixed']


## Step 5. The 80/20 split and CSV file

I used a stratified 80/20 split with a fixed random seed so that each intent is represented in both sets. The complete labelled dataset is saved as `intents.csv`.

In [3]:
train_df, test_df = train_test_split(
    df, test_size=0.20, random_state=42, stratify=df["intent"]
)
print("Training rows:", len(train_df))
print("Test rows:", len(test_df))
print("Training proportions:")
print(train_df["intent"].value_counts().sort_index().to_string())
print("Test proportions:")
print(test_df["intent"].value_counts().sort_index().to_string())

Training rows: 144
Test rows: 36
Training proportions:
intent
booking_cancellation    24
booking_modification    24
hotel_amenities         24
out_of_scope            24
room_availability       24
room_booking            24
Test proportions:
intent
booking_cancellation    6
booking_modification    6
hotel_amenities         6
out_of_scope            6
room_availability       6
room_booking            6


## Step 6. Baseline: provisional keyword rules on the test set

I tested the provisional keyword rules written in this notebook against the 36-message test set. This is a baseline for the prototype only. It is not a test of my original Portfolio Project 1 chatbot because I did not verify that project's source code or rules.

In [4]:
def provisional_rule_classifier(message):
    text = message.lower()
    rules = [
        ("booking_cancellation", r"\bcancel|cancell|hagarika|guhagarika|gusiba|isibwa|nyihagarike|withdraw|terminate\b"),
        ("booking_modification", r"\bchange|modify|update|amend|move my|extend|shift|guhindura|kongera|edit|add one more\b"),
        ("room_availability", r"\bavailable|availability|vacan|free|open|iraboneka|zihari|ihari|vacancy|fully booked|check avail\b"),
        ("hotel_amenities", r"\bwi-?fi|wifi|internet|pool|piscine|breakfast|parking|restaurant|gym|facilit|laundry|shuttle|air conditioning|service|towels|elevator|lift|hot water|salle de sport|generator|tv\b"),
        ("room_booking", r"\bbook|booking|reserve|reservation|fata|kubika|kubookinga|bika|room for|accommodation\b"),
    ]
    for label, pattern in rules:
        if re.search(pattern, text):
            return label
    return "out_of_scope"

test_df = test_df.copy()
test_df["prediction"] = test_df["text"].map(provisional_rule_classifier)
correct = int((test_df["intent"] == test_df["prediction"]).sum())
total = len(test_df)
accuracy = correct / total
print(f"Correct: {correct}/{total}")
print(f"Accuracy: {accuracy:.1%}")
errors = test_df[test_df["intent"] != test_df["prediction"]]
print("Incorrect predictions:")
for _, row in errors.iterrows():
    print(f"- Text: {row['text']!r} | Actual: {row['intent']} | Predicted: {row['prediction']}")

Correct: 32/36
Accuracy: 88.9%
Incorrect predictions:
- Text: 'Is there free parking for guests?' | Actual: hotel_amenities | Predicted: room_availability
- Text: 'Ese mufite chambre iboneka ku Tuesday?' | Actual: room_availability | Predicted: out_of_scope
- Text: 'Can u delete my hotel reservation?' | Actual: booking_cancellation | Predicted: room_booking
- Text: 'Please remove my booking from the system.' | Actual: booking_cancellation | Predicted: booking_modification


## Step 7. Entities of the hotel domain

I used regular expressions as a simple entity ruler for hotel-related information. These patterns demonstrate entity matching; they are not a trained entity-recognition model. The counts represent matched mentions across the dataset, not the number of unique messages containing each entity.

In [5]:
ENTITY_PATTERNS = {
    "LOCATION": r"\b(Kigali|Musanze|Huye|Rubavu|Rwanda)\b",
    "ROOM_TYPE": r"\b(standard|single|double|twin|queen|family)\s+room\b|\b(single|double|twin|queen|family)\b",
    "DATE": r"\b(Monday|Tuesday|Friday|Saturday|tomorrow|tonight|weekend|next week|next month|2026-\d{2}-\d{2})\b",
    "GUEST_COUNT": r"\b(one|two|three|four|1|2|3|4)\s+(guests?|people|pax)\b|\b\d+\s+pax\b",
}
entity_counts = {}
for entity, pattern in ENTITY_PATTERNS.items():
    entity_counts[entity] = sum(
        len(re.findall(pattern, message, flags=re.IGNORECASE))
        for message in df["text"]
    )
print("Entity mention counts:")
for entity, count in entity_counts.items():
    print(f"{entity}: {count}")

Entity mention counts:
LOCATION: 2
ROOM_TYPE: 21
DATE: 44
GUEST_COUNT: 10


## Step 8. Publish and submit

The dataset folder contains the three assignment deliverables:

1. `Assignment_8_Home_Intent_Dataset.ipynb` – this notebook.
2. `intents.csv` – the labelled synthetic dataset.
3. `README.md` – the dataset report.

### Remaining limitations

- The dataset contains 0 genuine-person messages; all 180 messages are synthetic.
- No UNMATCHED log or Step 10 local-phrasing records were available for this prototype.
- The original Portfolio Project 1 Hotel Booking Chatbot code and rules have not been verified.
- The baseline result measures only the provisional keyword rules in this notebook.
- The dataset and its results therefore represent a prototype, not a verified reproduction of the original Portfolio Project 1 chatbot.